# 08 - Results analysis

This notebook reads the evaluation files we downloaded from Kaggle (notebooks 01 and 07) and builds every table used in the report.

All five models are tested on the same 500 problems (MATH-500) with greedy decoding and `repetition_penalty=1.1`:

| Name | What it is |
|---|---|
| base | Qwen3.5-2B-Base, no fine-tuning |
| cot | fine-tuned on one correct reasoning chain (conventional reasoning) |
| refine | fine-tuned to write 2 attempts and then a rewritten final solution (ablation B) |
| review | fine-tuned to write 2 attempts, a short critique, then the final answer (method C, critiques by Qwen3.5-9B) |
| review_27b | same as review, but the critiques were written by the bigger Qwen3.8-27B teacher |

In [1]:
import json
from math import comb
from pathlib import Path
from collections import Counter

import pandas as pd

In [2]:
folder=Path('../experiments/08_eval_rp')

files={
    'base': folder/'qwen2b_math500_baseline_rp1.1.json',
    'cot': folder/'eval_cot_rp1.1_annotated.json',
    'refine': folder/'eval_refine_rp1.1_annotated.json',
    'review': folder/'eval_review_rp1.1_annotated.json',
    'review_27b': folder/'eval_review_27b_rp1.1_annotated.json',
}

results={}

for name,path in files.items():
    with path.open('r',encoding='utf-8') as file:
        results[name]=json.load(file)

for name,records in results.items():
    print(name,len(records))

base 500
cot 500
refine 500
review 500
review_27b 500


The base model file has a different format (it was made by notebook 01), so we rename its fields to match the others:
- `greedy_ok` becomes `final_ok` (was the answer right?)
- `no_box` tells us the model never wrote a `\boxed{}` answer, so `finished = not no_box`

In [3]:
for record in results['base']:
    record['final_ok']=record['greedy_ok']
    record['finished']=not record['no_box']

results['base'][0].keys()

dict_keys(['id', 'level', 'gold', 'greedy_pred', 'greedy_ok', 'maj_pred', 'maj_ok', 'sample_ok', 'no_box', 'greedy_text', 'sample_texts', 'final_ok', 'finished'])

## 1. Main accuracy table

- **accuracy**: how many of the 500 problems the final answer is correct
- **finished**: how often the model ended properly with an answer (did not run out of tokens or loop)

In [4]:
def accuracy(records: list)->float:
    return sum(r['final_ok'] for r in records)/len(records)

def finished_rate(records: list)->float:
    return sum(r['finished'] for r in records)/len(records)

table=[]

for name,records in results.items():
    table.append({
        'model': name,
        'correct': sum(r['final_ok'] for r in records),
        'accuracy %': round(100*accuracy(records),1),
        'finished %': round(100*finished_rate(records),1),
    })

pd.DataFrame(table)

,model,correct,accuracy %,finished %
0,base,278,55.6,91.0
1,cot,287,57.4,94.8
2,refine,273,54.6,86.0
3,review,299,59.8,89.8
4,review_27b,306,61.2,90.0


## 2. Does the model really correct itself?

Only the refine and review models write two attempts before the final answer, so this part is only for them.

- **first %**: accuracy of `[Answer 1]`, the model's first attempt
- **final %**: accuracy of `[Final answer]`, after reconsidering
- **gain**: final minus first. Positive means reconsidering helped
- **wrong to right**: first attempt wrong, final answer right (a real fix)
- **right to wrong**: first attempt right, final answer wrong (the model talked itself out of a right answer)

For the two flip numbers we only count problems where the first attempt has a readable `\boxed{}` answer, otherwise we cannot tell if it was right.

In [5]:
two_step=['refine','review','review_27b']

table=[]

for name in two_step:
    records=results[name]
    readable=[r for r in records if r['first_answer']!=None]
    n=len(records)

    table.append({
        'model': name,
        'first %': round(100*sum(r['first_ok'] for r in records)/n,1),
        'final %': round(100*sum(r['final_ok'] for r in records)/n,1),
        'gain': round(100*(sum(r['final_ok'] for r in records)-sum(r['first_ok'] for r in records))/n,1),
        'wrong to right': sum((not r['first_ok']) and r['final_ok'] for r in readable),
        'right to wrong': sum(r['first_ok'] and (not r['final_ok']) for r in readable),
    })

pd.DataFrame(table)

,model,first %,final %,gain,wrong to right,right to wrong
0,refine,56.8,54.6,-2.2,44,55
1,review,57.4,59.8,2.4,36,24
2,review_27b,57.4,61.2,3.8,41,22


## 3. Can the model pick the right attempt?

The most useful moment for reconsidering is when the two attempts disagree and **exactly one** of them is right.
If the model just guessed, it would pick the right one about 50% of the time.

In [6]:
def exactly_one_right(record: dict)->bool:
    if record['first_answer']==None or record['second_answer']==None:
        return False
    return record['first_ok']!=record['second_ok']

table=[]

for name in two_step:
    cases=[r for r in results[name] if exactly_one_right(r)]
    picked=sum(r['final_ok'] for r in cases)

    table.append({
        'model': name,
        'cases': len(cases),
        'picked right': picked,
        'picked right %': round(100*picked/len(cases),1),
    })

pd.DataFrame(table)

,model,cases,picked right,picked right %
0,refine,130,85,65.4
1,review,108,77,71.3
2,review_27b,111,85,76.6


## 4. Accuracy by difficulty level

MATH problems have levels 1 (easy) to 5 (hard, competition level).

In [7]:
levels=sorted(set(r['level'] for r in results['base']))

table=[]

for name,records in results.items():
    row={'model': name}
    for level in levels:
        subset=[r for r in records if r['level']==level]
        row[level]=round(100*accuracy(subset),1)
    table.append(row)

pd.DataFrame(table)

,model,1,2,3,4,5
0,base,88.4,81.1,64.8,48.4,27.6
1,cot,86.0,78.9,71.4,47.7,32.1
2,refine,86.0,78.9,71.4,48.4,20.9
3,review,86.0,81.1,74.3,50.8,34.3
4,review_27b,93.0,76.7,78.1,55.5,32.8


## 5. Are the differences real or just luck? (McNemar test)

Two models can differ by a few points just by luck. The **McNemar test** checks if the difference is real.

Idea: compare two models A and B on the same 500 problems. Problems that both get right, or both get wrong, tell us nothing about which is better. Only the problems where they **disagree** matter:

- **b** = problems A gets right but B gets wrong
- **c** = problems B gets right but A gets wrong

If A and B were equally good, each disagreement would be like a coin flip, so b and c should be roughly equal.

Example: suppose they disagree on 10 problems. A splits them 6 to 4: very possible with a fair coin, so no evidence. A splits them 10 to 0: very unlikely by luck, so A is really better.

The **p-value** is the chance of seeing a split at least this uneven if the two models were really equal. We use the usual rule:
- p < 0.05 means **significant**: the difference is very unlikely to be luck
- p >= 0.05 means **not significant**: it could be luck, so we do not claim it

The p-value is computed exactly with the binomial formula: the chance of getting k or fewer heads out of n fair coin flips is `sum of comb(n, j) / 2**n for j = 0..k`, doubled because the difference could go either way.

In [8]:
def coin_flip_p_value(b: int, c: int)->float:
    n=b+c
    k=min(b,c)
    p=sum(comb(n,j) for j in range(k+1))/2**n
    return min(1.0,2*p)

def mcnemar(name_a: str, name_b: str)->dict:
    a={r['id']: r['final_ok'] for r in results[name_a]}
    b={r['id']: r['final_ok'] for r in results[name_b]}

    a_only=sum(a[i] and not b[i] for i in a)
    b_only=sum(b[i] and not a[i] for i in a)
    p=coin_flip_p_value(a_only,b_only)

    return {
        'A': name_a,
        'B': name_b,
        'A right, B wrong': a_only,
        'B right, A wrong': b_only,
        'p-value': round(p,4),
        'significant': p<0.05,
    }

print(mcnemar('review_27b','base'))

{'A': 'review_27b', 'B': 'base', 'A right, B wrong': 68, 'B right, A wrong': 40, 'p-value': 0.0091, 'significant': True}


In [9]:
comparisons=[
    ('review_27b','base'),
    ('review_27b','cot'),
    ('review_27b','refine'),
    ('review_27b','review'),
    ('review','base'),
    ('review','cot'),
    ('review','refine'),
    ('cot','base'),
    ('refine','base'),
]

pd.DataFrame([mcnemar(a,b) for a,b in comparisons])

,A,B,"A right, B wrong","B right, A wrong",p-value,significant
0,review_27b,base,68,40,0.0091,True
1,review_27b,cot,62,43,0.0785,False
2,review_27b,refine,71,38,0.0020,True
3,review_27b,review,48,41,0.5250,False
4,review,base,69,48,0.0640,False
5,review,cot,65,53,0.3112,False
6,review,refine,75,49,0.0244,True
7,cot,base,60,51,0.4478,False
8,refine,base,55,60,0.7093,False


## 6. Is picking the right attempt better than guessing?

Same coin-flip idea as above: if the model guessed, each "exactly one right" case is a 50/50 coin flip.
We check how unlikely the observed number of right picks would be under pure guessing.

In [10]:
def better_than_guess_p_value(picked: int, cases: int)->float:
    p=sum(comb(cases,j) for j in range(picked,cases+1))/2**cases
    return min(1.0,2*p)

for name in two_step:
    cases=[r for r in results[name] if exactly_one_right(r)]
    picked=sum(r['final_ok'] for r in cases)
    print(name,'| picked',picked,'of',len(cases),'| p-value vs guessing:',better_than_guess_p_value(picked,len(cases)))

refine | picked 85 of 130 | p-value vs guessing: 0.0005715227032055847
review | picked 77 of 108 | p-value vs guessing: 1.1153098221083039e-05
review_27b | picked 85 of 111 | p-value vs guessing: 1.6986304619987468e-08


## 7. Effect of the repetition penalty

The first evaluation (notebook 07, `repetition_penalty=1.0`) showed the fine-tuned models sometimes repeat the same lines until they run out of tokens.
Re-running with `repetition_penalty=1.1` reduced this. Files for 1.0 are in `experiments/07_eval`.

In [11]:
old_folder=Path('../experiments/07_eval')

table=[]

for name in ['cot','refine','review']:
    with (old_folder/f'eval_{name}.json').open('r',encoding='utf-8') as file:
        old=json.load(file)
    new=results[name]

    table.append({
        'model': name,
        'accuracy % (rp 1.0)': round(100*accuracy(old),1),
        'accuracy % (rp 1.1)': round(100*accuracy(new),1),
        'finished % (rp 1.0)': round(100*finished_rate(old),1),
        'finished % (rp 1.1)': round(100*finished_rate(new),1),
    })

pd.DataFrame(table)

,model,accuracy % (rp 1.0),accuracy % (rp 1.1),finished % (rp 1.0),finished % (rp 1.1)
0,cot,59.0,57.4,90.8,94.8
1,refine,55.6,54.6,85.4,86.0
2,review,57.2,59.8,82.8,89.8


## 8. Examples of self-correction

Problems where the review_27b model's **first attempt was wrong**, the review found the mistake, and the **final answer was right**.

In [12]:
def get_review(text: str)->str:
    if '[Review]' not in text:
        return ''
    return text.split('[Review]')[1].split('[Final answer]')[0].strip()

fixed=[r for r in results['review_27b'] if exactly_one_right(r) and (not r['first_ok']) and r['final_ok']]
fixed=sorted(fixed,key=lambda r: r['n_tokens'])

print(len(fixed),'fixed cases')

for r in fixed[:3]:
    print('='*80)
    print(r['level'],'| gold:',r['gold'],'| first answer:',r['first_answer'],'| final answer:',r['final_answer'])
    print('Review:',get_review(r['text']))

39 fixed cases
1 | gold: 1600 | first answer: 1800 | final answer: 1600
Review: In my first attempt, I calculated the middle term as 798 but incorrectly added it to 441 and 361, resulting in a final answer of 1800. In my second attempt, I also found the middle term to be 798, but I correctly summed all three components—441, 798, and 361—to get 1600. The mistake in the first solution occurred during the final addition step, where I failed to include the constant term 361 in the total sum. By properly adding $441 + 798 + 361$ in the second attempt, I arrived at the correct value. Therefore the final answer is \boxed{1600}.
3 | gold: 23 | first answer: 143 | final answer: 23
Review: In my first attempt, I incorrectly calculated the GCF as 132 because I mistakenly thought that dividing 6432 by 132 yielded zero without checking if there were any smaller common divisors, leading me to the final answer of 143. In my second attempt, I correctly identified the prime factorizations of both numbe

## 9. Bigger model: the same experiment with Qwen3.5-4B-Base

We repeated the whole pipeline with the 4B model (notebooks 01, 02, 03_4b, 04b, 05_4b, 06, 07 with `model_name='Qwen/Qwen3.5-4B-Base'`):
the 4B model wrote its own attempts, the 27B teacher reviewed them, and we trained a review model and a CoT model on the same problems.

In [13]:
files_4b={
    'base_4b': folder/'qwen4b_math500_baseline_rp1.1.json',
    'cot_4b': folder/'eval_cot_4b_rp1.1_annotated.json',
    'review_4b': folder/'eval_review_4b_rp1.1_annotated.json',
}

for name,path in files_4b.items():
    with path.open('r',encoding='utf-8') as file:
        results[name]=json.load(file)

for record in results['base_4b']:
    record['final_ok']=record['greedy_ok']
    record['finished']=not record['no_box']

table=[]

for name in ['base','cot','review_27b','base_4b','cot_4b','review_4b']:
    records=results[name]
    table.append({
        'model': name,
        'accuracy %': round(100*accuracy(records),1),
        'finished %': round(100*finished_rate(records),1),
    })

pd.DataFrame(table)

,model,accuracy %,finished %
0,base,55.6,91.0
1,cot,57.4,94.8
2,review_27b,61.2,90.0
3,base_4b,68.2,95.6
4,cot_4b,71.6,98.6
5,review_4b,75.4,96.2


In [14]:
table=[]

for name in ['review_27b','review_4b']:
    records=results[name]
    readable=[r for r in records if r['first_answer']!=None]
    cases=[r for r in records if exactly_one_right(r)]
    n=len(records)

    table.append({
        'model': name,
        'first %': round(100*sum(r['first_ok'] for r in records)/n,1),
        'final %': round(100*sum(r['final_ok'] for r in records)/n,1),
        'wrong to right': sum((not r['first_ok']) and r['final_ok'] for r in readable),
        'right to wrong': sum(r['first_ok'] and (not r['final_ok']) for r in readable),
        'picked right %': round(100*sum(r['final_ok'] for r in cases)/len(cases),1),
    })

pd.DataFrame(table)

,model,first %,final %,wrong to right,right to wrong,picked right %
0,review_27b,57.4,61.2,41,22,76.6
1,review_4b,68.2,75.4,50,14,83.1


In [15]:
table=[]

for name in ['base_4b','cot_4b','review_4b']:
    row={'model': name}
    for level in levels:
        subset=[r for r in results[name] if r['level']==level]
        row[level]=round(100*accuracy(subset),1)
    table.append(row)

pd.DataFrame(table)

,model,1,2,3,4,5
0,base_4b,86.0,87.8,82.9,68.0,38.1
1,cot_4b,97.7,88.9,82.9,71.1,43.3
2,review_4b,90.7,91.1,88.6,72.7,52.2


In [16]:
comparisons_4b=[
    ('review_4b','base_4b'),
    ('review_4b','cot_4b'),
    ('cot_4b','base_4b'),
]

pd.DataFrame([mcnemar(a,b) for a,b in comparisons_4b])

,A,B,"A right, B wrong","B right, A wrong",p-value,significant
0,review_4b,base_4b,63,27,0.0002,True
1,review_4b,cot_4b,51,32,0.0475,True
2,cot_4b,base_4b,50,33,0.0784,False


## 10. Summary

- Fine-tuning the model to **critique its own two attempts** (review) gives the best accuracy at both sizes.
- 2B: review (27B teacher) 61.2% vs base 55.6% (significant) and vs CoT 57.4% (not significant).
- 4B: review 75.4% vs base 68.2% and vs CoT 71.6%, **both significant**. With a bigger model, self-refinement clearly beats conventional reasoning.
- The model **really reconsiders**: the final answer is better than its own first attempt, it fixes more answers than it breaks, and it picks the right attempt far more often than guessing.
- **Critiquing beats rewriting**: the refine model (rewrites the whole solution) gets worse after reconsidering.
- **A stronger teacher helps** (review_27b > review), but that difference alone is not significant.